# Veri Madenciliği Projesi
## Adım 1 — Ham Dosyaların Birleştirilmesi (Data Merging)
**Veri Seti:** EDM 2023 — Online Öğrenci Etkileşim Logları  
**Amaç:** `assignment_details.csv`, `sequence_details.csv`, `action_logs.csv` ve `training_unit_test_scores.csv` dosyalarını birleştirerek tek bir analiz verisi (`edm_2023.csv`) oluşturmak.

---
### Veri Seti Genel Yapısı
| Dosya | İçerik |
|---|---|
| `assignment_details.csv` | Ödev meta verileri (başlangıç/bitiş zamanı, sequence_id) |
| `sequence_details.csv` | Öğrenme sekansları (sequence türü, içerik bilgisi) |
| `action_logs.csv` | Öğrenci etkileşim logları (her hamle, her tıklama) |
| `training_unit_test_scores.csv` | Öğrencilerin problem bazında ham skorları |

Bu dört tablo birbirine yabancı anahtar ilişkileriyle bağlıdır:  
`assignment_details ↔ sequence_details` → `sequence_id`  
`merged_df ↔ action_logs` → `assignment_log_id`  
`full_data_df ↔ training_scores` → `assignment_log_id + problem_id`

---
## 1.1 Kütüphane Kurulumu ve İlk Birleştirme

`polars` kütüphanesi, büyük log verilerini pandas'a kıyasla çok daha hızlı işlediği için tercih edilmiştir.  
`ignore_errors=True` parametresi, `assignment_start_time` sütunundaki bazı hatalı tarih formatlarının parse hatası vermeden geçilmesini sağlar.

**Birleştirme Stratejisi:**  
- `assignment_details` ← **inner join** → `sequence_details` (sequence_id üzerinden): Yalnızca eşleşen kayıtlar tutulur; sequence bilgisi olmayan ödevler analizde anlamsız olduğundan dışlanır.

In [ ]:
# Ortaya çıkabilecek hataları yok sayarak 'assignment_details.csv' ve 'sequence_details.csv' dosyalarını yükleyeceğiz

!pip install polars
import polars as pl


assignment_df = pl.read_csv(
    'assignment_details.csv',
    schema_overrides={'assignment_start_time': pl.String},
    ignore_errors=True
)
sequence_df = pl.read_csv('sequence_details.csv', ignore_errors=True)

# 'sequence_id' sütununu kullanarak dosyaları birleştireceğiz
merged_df = assignment_df.join(sequence_df, on='sequence_id', how='inner')

# İLk 5 satırın çıktısına bakacağız
print(merged_df.head())

shape: (5, 16)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ assignmen ┆ teacher_i ┆ class_id  ┆ student_i ┆ … ┆ sequence_ ┆ sequence_ ┆ sequence_ ┆ sequence │
│ t_log_id  ┆ d         ┆ ---       ┆ d         ┆   ┆ folder_pa ┆ folder_pa ┆ name      ┆ _problem │
│ ---       ┆ ---       ┆ str       ┆ ---       ┆   ┆ th_level_ ┆ th_level_ ┆ ---       ┆ _ids     │
│ str       ┆ str       ┆           ┆ str       ┆   ┆ 4         ┆ 5         ┆ str       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆ ---       ┆ ---       ┆           ┆ str      │
│           ┆           ┆           ┆           ┆   ┆ str       ┆ str       ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2PLEB2KWK ┆ 22OEQXISY ┆ 133F5L5O9 ┆ L97DTM607 ┆ … ┆ Topic A-- ┆ null      ┆ Problem   ┆ [22SGPX3 │
│ 9         ┆ V         ┆ 5         ┆           ┆   ┆ -Lesson 1 ┆           

---
## 1.2 Action Loglarının Eklenmesi ve Eksik Problem ID Temizliği

`action_logs.csv`, öğrencilerin her adımını (deneme, ipucu isteme vb.) saniye bazında kaydeden ana etkileşim tablosudur.  
Bu tablo `assignment_log_id` üzerinden mevcut birleştirilmiş tabloya eklenir.

**Neden problem_id NULL satırları silinir?**  
`problem_id` NULL olan satırlar, öğrencinin bir problemi çözmeye çalışmadığı duraksamalı aksiyonlara (örn. sayfa açma, menü gezme) karşılık gelir. Başarı tahmini açısından bu satırların skor bilgisi olmayacağından ve hedef değişkeni kirleteceklerinden temizlenmeleri zorunludur.

In [ ]:
# 'action_logs.csv' dosyasını yükleyeceğiz
action_logs_df = pl.read_csv('action_logs.csv')

# Yüklediğimiz dosyayı yukarıda oluşturduğumuz 'merged_df' dosyasıyla 'assignment_log_id' sütununu kullanarak birleştireceğiz
final_merged_df = merged_df.join(action_logs_df, on='assignment_log_id', how='inner')

# 'problem_id' sütununda eksik veri bulunan satırları sileceğiz 
final_merged_df = final_merged_df.drop_nulls(subset=['problem_id'])

# Daha sonraki analizlerde kullanılmak üzere elimizdeki veri setini kopyalayacağız
full_data_df = final_merged_df.clone()

---
## 1.3 Eksik Değer ve Benzersiz Değer Kontrolü (Ham Veri)

Birleştirme sonrası veri kalitesini doğrulamak için:
1. Her sütundaki `NULL` sayısı kontrol edilir — bu sayının sıfıra yakın olması birleştirmenin doğru çalıştığının göstergesidir.
2. Her sütundaki benzersiz değer sayısı kontrol edilir — kardinalite analizi, hangi sütunların kategorik, hangilerinin numerik davranacağını anlamaya yardımcı olur.

In [ ]:
# Son veri setindeki eksik verileri kontrol edeceğiz
print(full_data_df.null_count())

# Her bir sütundaki benzersiz değerlerin sayısına bakacağız
for column in full_data_df.columns:
    unique_count = full_data_df.select(pl.col(column).n_unique()).item()
    print(f"Column '{column}': {unique_count} unique values")

shape: (1, 25)
┌────────────┬────────────┬──────────┬────────────┬───┬────────────┬────────┬──────────┬───────────┐
│ assignment ┆ teacher_id ┆ class_id ┆ student_id ┆ … ┆ continuous ┆ action ┆ hint_id  ┆ explanati │
│ _log_id    ┆ ---        ┆ ---      ┆ ---        ┆   ┆ _score_vie ┆ ---    ┆ ---      ┆ on_id     │
│ ---        ┆ u32        ┆ u32      ┆ u32        ┆   ┆ wable      ┆ u32    ┆ u32      ┆ ---       │
│ u32        ┆            ┆          ┆            ┆   ┆ ---        ┆        ┆          ┆ u32       │
│            ┆            ┆          ┆            ┆   ┆ u32        ┆        ┆          ┆           │
╞════════════╪════════════╪══════════╪════════════╪═══╪════════════╪════════╪══════════╪═══════════╡
│ 0          ┆ 0          ┆ 0        ┆ 0          ┆ … ┆ 12634649   ┆ 0      ┆ 17842596 ┆ 17895102  │
└────────────┴────────────┴──────────┴────────────┴───┴────────────┴────────┴──────────┴───────────┘
Column 'assignment_log_id': 638201 unique values
Column 'teacher_id': 2033 u

---
## 1.4 Bellek Optimize Lazy Execution ile Yeniden Birleştirme

Polars'ın **lazy API**'si, tüm veriyi belleğe yüklemeden önce sorgu planını optimize eder.  
Bu yaklaşım özellikle 1.7 milyon satır gibi büyük veri setlerinde bellek tüketimini ve işlem süresini önemli ölçüde düşürür.  

`lazy_plan.collect()` çağrısı yalnızca bu noktada verinin tamamını belleğe alır — bu "lazy evaluation" prensibinin temelidir.

In [ ]:
# Belleği zorlamamak için geçici plan (lazy_plan) oluşturacağız
lazy_plan = (
    pl.scan_csv(
        'assignment_details.csv',
        schema_overrides={'assignment_start_time': pl.String},
        ignore_errors=True
    )
    .join(pl.scan_csv('sequence_details.csv', ignore_errors=True), on='sequence_id', how='inner')
    .join(pl.scan_csv('action_logs.csv'), on='assignment_log_id', how='inner')
    .filter(pl.col('problem_id').is_not_null())
)

# Planı çalıştıracağız ve sonuçları alacağız
full_data_df = lazy_plan.collect()

---
## 1.5 Skor Dosyasının Yüklenmesi

`training_unit_test_scores.csv`, öğrencilerin her problem için aldığı ham skoru (0 = yanlış, 1 = doğru) içerir.  
Bu tablo birleştirmenin son adımı için hazırlanır.

In [ ]:
training_scores_df = pl.read_csv('training_unit_test_scores.csv')
print(training_scores_df.head())

shape: (5, 4)
┌─────┬───────────────────┬────────────┬───────┐
│     ┆ assignment_log_id ┆ problem_id ┆ score │
│ --- ┆ ---               ┆ ---        ┆ ---   │
│ i64 ┆ str               ┆ str        ┆ i64   │
╞═════╪═══════════════════╪════════════╪═══════╡
│ 0   ┆ KMQMUDOYN         ┆ 9OLB1D1BS  ┆ 1     │
│ 1   ┆ KMQMUDOYN         ┆ 9OLB1D1BS  ┆ 0     │
│ 2   ┆ KMQMUDOYN         ┆ 9OLB1D1BS  ┆ 0     │
│ 3   ┆ KMQMUDOYN         ┆ 9OLB1D1BS  ┆ 1     │
│ 4   ┆ KMQMUDOYN         ┆ 9OLB1D1BS  ┆ 1     │
└─────┴───────────────────┴────────────┴───────┘


---
## 1.6 Skor Sütununun Ana Tabloya Eklenmesi

Skor tablosu, `assignment_log_id` **ve** `problem_id` çifti üzerinden birleştirilir.  
Neden çift anahtar kullanılır? Tek `assignment_log_id` ile birleştirilseydi, aynı ödevdeki farklı problemlerin skorları çakışabilirdi.  
**Inner join** kullanılmasının nedeni:** Yalnızca skoru olan etkileşim kayıtları hedef değişken oluşturmada kullanılabilir; skorlanmamış aksiyonlar bu aşamada kapsam dışı bırakılır.

In [ ]:
# training_scores_df tablosundan yalnızca ilgili sütunları seçeceğiz 
training_scores_df = training_scores_df.select('assignment_log_id', 'problem_id', 'score')

# full_data_df dosyasını training_scores_df dosyasıyla 'assignment_log_id' ve 'problem_id' üzerinden birleştireceğiz
# Yalnızca puanların bulunduğu kayıtları tutmak için iç birleştirme kullanılıyor
final_merged_df_with_scores = full_data_df.join(
    training_scores_df,
    on=['assignment_log_id', 'problem_id'],
    how='inner'
)

# Display the first 5 rows of the new merged DataFrame
print(final_merged_df_with_scores.head())

shape: (5, 26)
┌────────────┬────────────┬────────────┬────────────┬───┬────────────┬─────────┬───────────┬───────┐
│ assignment ┆ teacher_id ┆ class_id   ┆ student_id ┆ … ┆ action     ┆ hint_id ┆ explanati ┆ score │
│ _log_id    ┆ ---        ┆ ---        ┆ ---        ┆   ┆ ---        ┆ ---     ┆ on_id     ┆ ---   │
│ ---        ┆ str        ┆ str        ┆ str        ┆   ┆ str        ┆ str     ┆ ---       ┆ i64   │
│ str        ┆            ┆            ┆            ┆   ┆            ┆         ┆ str       ┆       │
╞════════════╪════════════╪════════════╪════════════╪═══╪════════════╪═════════╪═══════════╪═══════╡
│ 1L60X9CLMJ ┆ 13FM9BH12Z ┆ 2K6V6GTPMC ┆ 1O4U5A5XC0 ┆ … ┆ problem_st ┆ null    ┆ null      ┆ 1     │
│            ┆            ┆            ┆            ┆   ┆ arted      ┆         ┆           ┆       │
│ 1L60X9CLMJ ┆ 13FM9BH12Z ┆ 2K6V6GTPMC ┆ 1O4U5A5XC0 ┆ … ┆ problem_st ┆ null    ┆ null      ┆ 1     │
│            ┆            ┆            ┆            ┆   ┆ arted      ┆      

---
## 1.7 Son Veri Kalite Kontrolü — Eksik Değerler

In [ ]:
null_counts = final_merged_df_with_scores.null_count()
for col_name in null_counts.columns:
    print(f"Column '{col_name}': {null_counts[col_name].item()} null values")

Column 'assignment_log_id': 0 null values
Column 'teacher_id': 0 null values
Column 'class_id': 0 null values
Column 'student_id': 0 null values
Column 'sequence_id': 0 null values
Column 'assignment_release_date': 0 null values
Column 'assignment_due_date': 713332 null values
Column 'assignment_start_time': 0 null values
Column 'assignment_end_time': 194600 null values
Column 'sequence_folder_path_level_1': 0 null values
Column 'sequence_folder_path_level_2': 0 null values
Column 'sequence_folder_path_level_3': 0 null values
Column 'sequence_folder_path_level_4': 0 null values
Column 'sequence_folder_path_level_5': 1745977 null values
Column 'sequence_name': 0 null values
Column 'sequence_problem_ids': 0 null values
Column 'timestamp': 0 null values
Column 'problem_id': 0 null values
Column 'max_attempts': 1293534 null values
Column 'available_core_tutoring': 1293534 null values
Column 'score_viewable': 1293534 null values
Column 'continuous_score_viewable': 1293534 null values
Column

---
## 1.8 Son Veri Kalite Kontrolü — Benzersiz Değer Sayıları

Her sütunun kardinalitesi kontrol edilerek veri tutarlılığı doğrulanır.

In [ ]:
print('--- Unique Value Counts ---')
for column in final_merged_df_with_scores.columns:
    unique_count = final_merged_df_with_scores.select(pl.col(column).n_unique()).item()
    print(f"Column '{column}': {unique_count} unique values")

--- Unique Value Counts ---
Column 'assignment_log_id': 15468 unique values
Column 'teacher_id': 95 unique values
Column 'class_id': 127 unique values
Column 'student_id': 1499 unique values
Column 'sequence_id': 670 unique values
Column 'assignment_release_date': 1069 unique values
Column 'assignment_due_date': 413 unique values
Column 'assignment_start_time': 15468 unique values
Column 'assignment_end_time': 12680 unique values
Column 'sequence_folder_path_level_1': 2 unique values
Column 'sequence_folder_path_level_2': 9 unique values
Column 'sequence_folder_path_level_3': 39 unique values
Column 'sequence_folder_path_level_4': 236 unique values
Column 'sequence_folder_path_level_5': 1 unique values
Column 'sequence_name': 670 unique values
Column 'sequence_problem_ids': 670 unique values
Column 'timestamp': 452256 unique values
Column 'problem_id': 7062 unique values
Column 'max_attempts': 3 unique values
Column 'available_core_tutoring': 4 unique values
Column 'score_viewable': 2 

---
## 1.9 Birleştirilmiş Verinin Dile Getirilmesi ve İndirilmesi

`edm_2023.csv` dosyası oluşturularak sonraki notebook aşamaları (EDA + Feature Engineering) için hazır hale getirilir.

In [ ]:
final_merged_df_with_scores.write_csv('edm_2023.csv')

In [ ]:
from google.colab import files
files.download('edm_2023.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>